<a href="https://colab.research.google.com/github/Tanvi2307/The-Price-Is-Right/blob/main/agent1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q transformers accelerate bitsandbytes peft huggingface_hub litellm python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.3 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from huggingface_hub import login
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import PeftModel
from litellm import completion

In [4]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [5]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [6]:
# Load the Tokenizer and the Model

MODEL_NAME = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
)
print("Model loaded successfully!")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model loaded successfully!


In [7]:
def generate(prompt: str) -> str:
    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=5
        )
    return tokenizer.decode(outputs[0])

In [8]:
result = generate(
    "Never gonna give you up, never gonna"
)

print(result)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|begin_of_text|>Never gonna give you up, never gonna let you down, never


RESTART SESSION

In [1]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "TanviAgarwal"
RUN_NAME = "2026-09-19_05.58.52-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
REVISION = "8a6c78d51f9c7ebf3b34be70f463644a358b4801"
FINETUNED_MODEL = f"{HF_USER}/{PROJECT_RUN_NAME}"
print(FINETUNED_MODEL)

TanviAgarwal/price-2026-09-19_05.58.52-lite


In [4]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

In [5]:
#Load the base model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

print("Base model loaded!")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Base model loaded!


In [6]:
#Load the fine-tuned LoRA adapter

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    FINETUNED_MODEL,
    revision=REVISION
)

print("Fine-tuned price model loaded!")

adapter_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 73.4MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Fine-tuned price model loaded!


In [7]:
def price(description: str) -> float:

    PREFIX = "Price is $"
    QUESTION = "What does this cost to the nearest dollar?"
    prompt = f"{QUESTION}\n\n{description}\n\n{PREFIX}"

    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = fine_tuned_model.generate(inputs, max_new_tokens=5)
    result = tokenizer.decode(outputs[0])

    contents = result.split("Price is $")[1] #Python splits the string wherever "Price is $" appears.So we get something like:["Question...\n", "599"]
    contents = contents.replace(",", "")

    match = re.search( #re.search() searches a string using a regular expression (regex).
        r"[-+]?\d*\.\d+|\d+",
        contents
    )
    # match is not directly the string "599".It is a Match object.To get the actual matched text, we use: match.group()
    return float(match.group()) if match else 0

In [8]:
description = (
    "Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio"
)

result = price(description)

print("Predicted price:", result)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Predicted price: 60.0


In [12]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [14]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""


class Preprocessor:
    def __init__(
        self,
        model_name="groq/openai/gpt-oss-20b",
        reasoning_effort="low"
    ):
        self.model_name = model_name
        self.reasoning_effort = reasoning_effort

    def messages_for(self, text: str) -> list[dict]:
        return [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": text
            }
        ]

    def preprocess(self, text: str) -> str:
        messages = self.messages_for(text)
        response = completion(
            messages=messages,
            model=self.model_name,
            reasoning_effort=self.reasoning_effort
        )
        return response.choices[0].message.content

In [15]:
preprocessor = Preprocessor()

text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

print(text)

Title: Quadcast HyperX USB‑C Condenser Mic  
Category: Electronics  
Brand: HyperX  
Description: The Quadcast mic delivers crystal‑clear audio via a simple USB‑C connection.  
Details: It offers studio‑quality condenser pickup, intuitive on‑board controls, and plug‑and‑play compatibility with any computer.


In [22]:
result = price("Apple iPhone 13, 128GB, excellent condition")

print(f"Predicted price: ${result:.2f}")

Predicted price: $350.00


In [20]:
!mkdir -p agents
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week8/agents/agent.py -O agents/agent.py

In [21]:
from agents.agent import Agent

class SpecialistAgent(Agent):

    name = "Specialist Agent"
    color = Agent.RED

    def __init__(self, pricer):
        self.log("Specialist Agent is initializing - connecting to local pricer")
        self.pricer = pricer

    def price(self, description: str) -> float:
        self.log("Specialist Agent is calling fine-tuned model")

        result = self.pricer(description)

        self.log(
            f"Specialist Agent completed - predicting ${result:.2f}"
        )

        return result

In [23]:
agent = SpecialistAgent(price)

result = agent.price("Apple iPhone 13, 128GB, excellent condition")

print(f"Final result: ${result:.2f}")

Final result: $350.00
